In [0]:
# Databricks notebook: Lightweight dashboard for manual notebook runs

from pyspark.sql.functions import col, min as spark_min, max as spark_max
from pyspark.sql import functions as F

# Load the job run timeline table
job_run_df = spark.table("system.lakeflow.job_run_timeline")

# Filter for runs from today (or adjust date filter)
runs_today_df = job_run_df.filter(col("period_start_time") >= F.current_date())

# Group by run_id to get start/end times per run
dashboard_df = (
    runs_today_df
    .groupby("run_id")
    .agg(
        spark_min("period_start_time").alias("start_time"),
        spark_max("period_end_time").alias("end_time"),
    )
    .withColumn(
        "duration_seconds",
        F.unix_timestamp("end_time") - F.unix_timestamp("start_time")
    )
    .orderBy(col("start_time").desc())
)

# Show the result
display(dashboard_df)